In [ ]:
import numpy as np
import pandas as pd
import joblib
from sklearn.metrics import classification_report, roc_auc_score,  make_scorer,  make_scorer, average_precision_score
from sklearn.utils.class_weight import compute_sample_weight
import xgboost as xgb
import matplotlib.pyplot as plt
from statsmodels.stats.outliers_influence import variance_inflation_factor
import torch.nn as nn
import torch.nn.functional as F
import torch
from sklearn.utils.class_weight import compute_class_weight
import json
import copy 
import gc
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
import os, glob, gc

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:

IBAN_COL   = 'Account'
TS_COL     = 'Timestamp'
LABEL_COL  = 'proxy_label' 
HOLDING_PSP_COL = 'To Bank'
DECLARING_PSP_COL = 'From Bank'
N          = 1

drop_cols = [LABEL_COL, IBAN_COL, TS_COL,'key_lag1','Unnamed: 0_lag1','fold_lag1','Amount Received_lag1', 'Amount Paid_lag1']


In [ ]:
out = pd.read_csv('/Users/cbrou/Documents/fraud_class_memoire/dataset/fake_fncrf.csv',on_bad_lines='warn', engine='python')
print(len(out['bank'].unique()))
out['bank'] = out['bank'].astype(str).str.strip()
out = out[out['bank'] != '70']

1635


In [4]:
combined = out

In [5]:
combined = combined.drop(columns=['Is Laundering', 'nb.currency','delta.t', 'currency.mismatch', 'is.self.transfer', 'is.intra.bank',
       'log.amount', 'is.round.amount', 'hour.of.day', 'day.of.week',
       'is.off.hours', 'nb.distinct.to.bank_cum', 'nb.distinct.from.bank_cum',
       'nb.distinct.payfmt_cum', 'top.1.holder.RC', 'top.1.holder.SC',
       'nb.iban.holder', 'nb.events.holder', 'top.1.declaring.RC',
       'top.1.declaring.SC', 'nb.iban.declaring', 'nb.events.declaring',
       'fan.out', 'fan.in', 'fan.ratio', 'key', 'y_pred'])
bank_col = 'bank'

In [6]:
def create_sliding(combined):
    # 1. Unify categorical dtype detection (object + category)
    cat_features = [c for c in combined.columns 
                    if c not in [IBAN_COL, TS_COL, LABEL_COL, 'proxy_label', 'Account.1']
                    and combined[c].dtype.name in ('category', 'object')]

    for c in cat_features:
        if combined[c].dtype.name != 'category':
            combined[c] = combined[c].astype('category')

    # 2. cont_features = everything else
    cont_features = [c for c in combined.columns 
                    if c not in [IBAN_COL, TS_COL, LABEL_COL, 'proxy_label', 'is_first_event', 'Account.1'] + cat_features]

    EVENT_FEATURES_ORDERED = cont_features + cat_features

    combined = combined.sort_values([IBAN_COL, TS_COL]).reset_index(drop=True)
    combined['is_first_event'] = combined.groupby(IBAN_COL).cumcount() == 0
    combined[cont_features] = combined[cont_features].fillna(0)

    for c in cat_features:
        if '__MISSING__' not in combined[c].cat.categories:
            combined[c] = combined[c].cat.add_categories('__MISSING__')
        combined[c] = combined[c].fillna('__MISSING__')

    # --- vectorized lag1 (replaces the per-group Python loop) ---
    lag = combined.groupby(IBAN_COL)[EVENT_FEATURES_ORDERED].shift(1)
    lag.columns = [f'{f}_lag1' for f in EVENT_FEATURES_ORDERED]

    out = pd.concat(
        [lag, combined[[LABEL_COL, IBAN_COL, TS_COL, 'is_first_event']]],
        axis=1
    )
    out = out.loc[~out['is_first_event']].reset_index(drop=True)
    # --- end vectorized block ---

    cat_lag_cols = [f'{c}_lag1' for c in cat_features]
    num_lag_cols = [c for c in out.columns
                    if c not in cat_lag_cols + [LABEL_COL, IBAN_COL, TS_COL, 'is_first_event']]

    for c in cat_lag_cols:
        out[c] = out[c].fillna('__MISSING__').astype(str).astype('category')
    for c in num_lag_cols:
        out[c] = pd.to_numeric(out[c], errors='coerce')

    return out.drop(columns='is_first_event'), num_lag_cols, cat_lag_cols, cat_features

def cumulative_nunique_vectorized(df, group_col, value_col):
    dup = df.duplicated(subset=[group_col, value_col])
    first_seen = (~dup).astype(int)
    return first_seen.groupby(df[group_col]).cumsum()

def get_item(bic, d, str_key):
    entry = d.get(str(bic) if not pd.isna(bic) else 'nan')
    return entry[str_key] if entry else float('nan')

def features_creator(dataset):
    dataset = dataset.sort_values(['Account', 'Timestamp']).reset_index(drop=True)
    dataset['Timestamp'] = pd.to_datetime(dataset['Timestamp'])

    acc_grp = dataset.groupby('Account')

    dataset['nb.currency'] = acc_grp.cumcount() + 1
    dataset['delta.t'] = acc_grp['Timestamp'].diff().dt.total_seconds().fillna(0)

    dataset['currency.mismatch'] = (dataset['Receiving Currency'] != dataset['Payment Currency']).astype(int)
    dataset['is.self.transfer']  = (dataset['Account'] == dataset['Account.1']).astype(int)
    dataset['is.intra.bank']     = (dataset['From Bank'] == dataset['To Bank']).astype(int)

    dataset['log.amount']      = np.log1p(dataset['Amount Paid'].astype(float))
    dataset['Amount Paid'] = pd.to_numeric(dataset['Amount Paid'], errors='coerce')
    dataset['is.round.amount'] = (dataset['Amount Paid'] % 100 == 0).astype(int)

    dataset['hour.of.day']  = dataset['Timestamp'].dt.hour
    dataset['day.of.week']  = dataset['Timestamp'].dt.dayofweek
    dataset['is.off.hours'] = dataset['Timestamp'].dt.hour.between(0, 5).astype(int)

    dataset['nb.distinct.to.bank_cum']   = cumulative_nunique_vectorized(dataset, 'Account', 'To Bank')
    dataset['nb.distinct.from.bank_cum'] = cumulative_nunique_vectorized(dataset, 'Account', 'From Bank')
    dataset['nb.distinct.payfmt_cum']    = cumulative_nunique_vectorized(dataset, 'Account', 'Payment Format')

    # --- Holding PSP (keyed by To Bank) ---
    dict_bic_holding = {}
    for x, obj in dataset.groupby('To Bank'):
        vc_2 = obj['Receiving Currency'].value_counts()
        vc_3 = obj['Payment Currency'].value_counts()
        dict_bic_holding[str(x)] = {
            'top_RC': vc_2.index[0],
            'top_SC': vc_3.index[0],
            'nb.events.holding': obj['Timestamp'].nunique(),
            'nb.iban.holding': obj['Account'].nunique()
        }

    # --- Declaring PSP (keyed by From Bank) ---
    dict_bic_declaring = {}
    for x, obj in dataset.fillna({'From Bank': 'UNKNW'}).groupby('From Bank'):
        vc_2 = obj['Receiving Currency'].value_counts()
        vc_3 = obj['Payment Currency'].value_counts()
        dict_bic_declaring[str(x)] = {
            'top_RC': vc_2.index[0],
            'top_SC': vc_3.index[0],
            'nb.events.declaring': obj['Timestamp'].nunique(),
            'nb.iban.declaring': obj['Account'].nunique()
        }

    fan_out = dataset.groupby('Account')['Account.1'].nunique().rename('fan.out')
    fan_in  = dataset.groupby('Account.1')['Account'].nunique().rename('fan.in')

    to_bank_str   = dataset['To Bank'].apply(lambda b: str(b) if not pd.isna(b) else 'nan')
    from_bank_str = dataset['From Bank'].apply(lambda b: str(b) if not pd.isna(b) else 'nan')

    dataset['top.1.holder.RC']  = to_bank_str.map(lambda k: get_item(k, dict_bic_holding, 'top_RC'))
    dataset['top.1.holder.SC']  = to_bank_str.map(lambda k: get_item(k, dict_bic_holding, 'top_SC'))
    dataset['nb.iban.holder']   = to_bank_str.map(lambda k: get_item(k, dict_bic_holding, 'nb.iban.holding'))
    dataset['nb.events.holder'] = to_bank_str.map(lambda k: get_item(k, dict_bic_holding, 'nb.events.holding'))

    dataset['top.1.declaring.RC']  = from_bank_str.map(lambda k: get_item(k, dict_bic_declaring, 'top_RC'))
    dataset['top.1.declaring.SC']  = from_bank_str.map(lambda k: get_item(k, dict_bic_declaring, 'top_SC'))
    dataset['nb.iban.declaring']   = from_bank_str.map(lambda k: get_item(k, dict_bic_declaring, 'nb.iban.declaring'))
    dataset['nb.events.declaring'] = from_bank_str.map(lambda k: get_item(k, dict_bic_declaring, 'nb.events.declaring'))

    del dict_bic_declaring, dict_bic_holding

    dataset['fan.out']   = dataset['Account'].map(fan_out).fillna(0)
    dataset['fan.in']    = dataset['Account'].map(fan_in).fillna(0)
    dataset['fan.ratio'] = dataset['fan.in'] / (dataset['fan.out'] + 1)

    dataset['key'] = dataset['Account']  # replicates original df.assign(key=key) at concat time

    return dataset


In [ ]:
combined = combined.sort_values(TS_COL).reset_index(drop=True)
combined[TS_COL] = pd.to_datetime(combined[TS_COL], format='mixed', errors='coerce')
n_bad = combined[TS_COL].isna().sum()
if n_bad:
    print(f"Dropping {n_bad} rows with unparseable timestamps")
combined = combined.dropna(subset=[TS_COL])
combined[LABEL_COL] = combined[LABEL_COL].map({'Fraudeur': 1, 'Faux Positif': 0}).astype(float)
print(combined[LABEL_COL].unique())
print(combined[LABEL_COL].dtype)
n_splits = 5
t_min, t_max = combined[TS_COL].min(), combined[TS_COL].max()
edges = pd.date_range(t_min, t_max, periods=n_splits + 2)  # +2 -> n_splits test windows + initial train seed

gap = pd.Timedelta(hours=1)  # tune to your delta.t scale

train_dfs = []
test_dfs = []

for fold in range(n_splits):
    train_end = edges[fold + 1]
    test_start = train_end + gap
    test_end = edges[fold + 2]

    train_df = combined[combined[TS_COL] < train_end].assign(fold=fold)
    test_df  = combined[(combined[TS_COL] >= test_start) & (combined[TS_COL] < test_end)].assign(fold=fold)

    train_dfs.append(train_df)
    test_dfs.append(test_df)

    print(f"Fold {fold}")
    print(f"  train: {train_df[TS_COL].min()} – {train_df[TS_COL].max()} ({len(train_df)} rows)")
    print(f"  test : {test_df[TS_COL].min()} – {test_df[TS_COL].max()} ({len(test_df)} rows)")
    print(f"  train fraud rate: {train_df[LABEL_COL].mean():.4f}")
    print(f"  test  fraud rate: {test_df[LABEL_COL].mean():.4f}")
    print()

train_df = train_dfs[0]
test_df  = test_dfs[0]
y_train = train_df[LABEL_COL]
y_test = test_df[LABEL_COL]

Dropping 1 rows with unparseable timestamps
[0. 1.]
float64
Fold 0
  train: 2022-10-14 13:03:00 – 2022-10-29 04:54:00 (405357 rows)
  test : 2022-10-29 05:55:00 – 2022-11-12 20:42:00 (184538 rows)
  train fraud rate: 0.0642
  test  fraud rate: 0.1048

Fold 1
  train: 2022-10-14 13:03:00 – 2022-11-12 20:42:00 (590659 rows)
  test : 2022-11-12 21:49:00 – 2022-11-27 12:35:00 (9524 rows)
  train fraud rate: 0.0769
  test  fraud rate: 0.8850

Fold 2
  train: 2022-10-14 13:03:00 – 2022-11-27 12:35:00 (600193 rows)
  test : 2022-11-27 13:39:00 – 2022-12-12 03:47:00 (3105 rows)
  train fraud rate: 0.0897
  test  fraud rate: 0.9069

Fold 3
  train: 2022-10-14 13:03:00 – 2022-12-12 03:47:00 (603332 rows)
  test : 2022-12-12 05:43:00 – 2022-12-26 20:02:00 (849 rows)
  train fraud rate: 0.0940
  test  fraud rate: 0.9482

Fold 4
  train: 2022-10-14 13:03:00 – 2022-12-26 20:02:00 (604182 rows)
  test : 2022-12-27 00:00:00 – 2023-01-09 12:27:00 (218 rows)
  train fraud rate: 0.0952
  test  fraud rate

In [ ]:
# 1. build bank_id from raw source, single source of truth

print("banks:", train_df[bank_col].nunique())  # must show ~1633

exclude = ['Is Laundering', 'Account', 'Account.1', 'Timestamp', 'y_pred', 'bank',
           'Amount Received', 'Amount Paid', 'key', LABEL_COL]

# ---- 1. compute full features once (needed to fix column lists + fit global cat encoder) ----
full_feats = features_creator(train_df.copy())
full_feats['delta.t'] = np.log1p(full_feats['delta.t'])

cat_cols = [c for c in full_feats.select_dtypes(include=['object', 'category']).columns
            if c not in exclude]
num_cols = [c for c in full_feats.select_dtypes(include=['number']).columns
            if c not in exclude]

for c in cat_cols:
    print(c, full_feats[c].nunique())


banks: 1633
From Bank 1633
To Bank 5552
Receiving Currency 15
Payment Currency 15
Payment Format 7
top.1.holder.RC 15
top.1.holder.SC 15
top.1.declaring.RC 15
top.1.declaring.SC 15


In [ ]:
cat_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)),
])
cat_pipe.fit(full_feats[cat_cols])

cat_cardinalities = [len(cats) for cats in cat_pipe.named_steps['encoder'].categories_]

num_imputer = SimpleImputer(strategy='median')
num_imputer.fit(full_feats[num_cols])
num_scaler = StandardScaler().fit(num_imputer.transform(full_feats[num_cols]))

del full_feats
gc.collect()

os.makedirs('client_data', exist_ok=True)

expected_width = len(num_cols) + len(cat_cols)
for f in glob.glob('client_data/*.pt'):
    try:
        x, _ = torch.load(f)
        if x.shape[1] != expected_width:
            raise ValueError(f"stale width {x.shape[1]} != expected {expected_width}")
    except Exception as e:
        print('corrupt/stale, removing:', f, '-', e)
        os.remove(f)

LABEL_TMP_COL = '__y_train__'

# 2. main loop, resumable
for bid, group in train_df.groupby(bank_col):
    if len(group) < 5:
        continue

    out_path = f'client_data/{bid}.pt'
    if os.path.exists(out_path):
        continue  # already done, skip

    group = group.copy()
    # attach the label before features_creator sorts the rows, so it travels
    # with each row instead of being re-applied by (now stale) original index
    group[LABEL_TMP_COL] = y_train.loc[group.index].values

    group = features_creator(group)
    group['delta.t'] = np.log1p(group['delta.t'])

    cat_x = cat_pipe.transform(group[cat_cols]).astype(np.float32)
    num_x = num_scaler.transform(num_imputer.transform(group[num_cols])).astype(np.float32)

    # continuous features first, categorical codes last — matches
    # LinearWithEmbeddings.forward(), which slices x[:, :num_cont] / x[:, num_cont:]
    x = torch.tensor(np.hstack([num_x, cat_x]), dtype=torch.float32)
    y = torch.tensor(group[LABEL_TMP_COL].values, dtype=torch.long)

    tmp_path = out_path + '.tmp'
    torch.save((x, y), tmp_path)
    os.replace(tmp_path, out_path)  # atomic on POSIX
    del x, y, cat_x, num_x, group

gc.collect()


1222

In [ ]:

CLASS_WEIGHTS = compute_class_weight('balanced', classes=np.array([0, 1]), y=y_train)
CLASS_WEIGHTS_T = torch.tensor(CLASS_WEIGHTS, dtype=torch.float32)

criterion = nn.CrossEntropyLoss(weight=CLASS_WEIGHTS_T)

def client_update(model, x, y, epochs=3, lr=0.01):
    opt = torch.optim.SGD(model.parameters(), lr=lr)
    for _ in range(epochs):
        opt.zero_grad()
        loss = criterion(model(x), y)  # y must be Long, shape (N,)
        loss.backward()
        opt.step()
    return model.state_dict()

# ---------- FedAvg ----------
def fedavg_aggregate(client_states, client_sizes):
    total = sum(client_sizes)
    new_state = {}
    for k in client_states[0]:
        new_state[k] = sum(
            client_states[i][k] * (client_sizes[i] / total)
            for i in range(len(client_states))
        )
    return new_state

# ---------- FedAdam ----------
class FedAdamServer:
    def __init__(self, global_state, lr=5e-4, beta1=0.9, beta2=0.99, eps=1e-3, clip_norm = 1.0):
        self.state = {k: v.clone() for k, v in global_state.items()}
        self.m = {k: torch.zeros_like(v) for k, v in global_state.items()}
        self.v = {k: torch.zeros_like(v) for k, v in global_state.items()}
        self.lr, self.b1, self.b2, self.eps = lr, beta1, beta2, eps
        self.t = 0
        self.C = clip_norm   

    def _clip_client_delta(self, client_delta):
        # client_delta: dict of tensors (this client's update = local_state - global_state)
        total_norm = torch.sqrt(sum((v ** 2).sum() for v in client_delta.values()))
        clip_coef = min(1.0, self.C / (total_norm + 1e-6))
        return {k: v * clip_coef for k, v in client_delta.items()}

    def step(self, client_states, client_sizes):
        L = len(client_states)
        clipped_deltas = []
        for cs in client_states:
            delta = {k: cs[k] - self.state[k] for k in self.state}
            clipped_deltas.append(self._clip_client_delta(delta))

        avg_delta = {
            k: sum(cd[k] for cd in clipped_deltas) / L
            for k in self.state
        }

        self.t += 1
        for k in self.state:
            self.m[k] = self.b1 * self.m[k] + (1 - self.b1) * avg_delta[k]
            self.v[k] = self.b2 * self.v[k] + (1 - self.b2) * (avg_delta[k] ** 2)
            m_hat = self.m[k] / (1 - self.b1 ** self.t)
            v_hat = self.v[k] / (1 - self.b2 ** self.t)
            self.state[k] += self.lr * m_hat / (v_hat.sqrt() + self.eps)
        return self.state

class FedAdamDPServer(FedAdamServer):
    def __init__(self, global_state, lr=5e-4, beta1=0.9, beta2=0.99, eps=1e-3,
                 clip_norm=1.0, noise_sigma=0.1):
        super().__init__(global_state, lr, beta1, beta2, eps)
        self.C = clip_norm       # per-client clipping threshold
        self.sigma = noise_sigma # noise multiplier

    def _clip_client_delta(self, client_delta):
        # client_delta: dict of tensors (this client's update = local_state - global_state)
        total_norm = torch.sqrt(sum((v ** 2).sum() for v in client_delta.values()))
        clip_coef = min(1.0, self.C / (total_norm + 1e-6))
        return {k: v * clip_coef for k, v in client_delta.items()}

    def step(self, client_states, client_sizes):
        L = len(client_states)
        clipped_deltas = []
        for cs in client_states:
            delta = {k: cs[k] - self.state[k] for k in self.state}
            clipped_deltas.append(self._clip_client_delta(delta))

        sum_delta = {k: sum(cd[k] for cd in clipped_deltas) for k in self.state}

        noisy_delta = {
            k: (sum_delta[k] + torch.randn_like(sum_delta[k]) * (self.sigma * self.C)) / L
            for k in self.state
        }

        self.t += 1
        for k in self.state:
            self.m[k] = self.b1 * self.m[k] + (1 - self.b1) * noisy_delta[k]
            self.v[k] = self.b2 * self.v[k] + (1 - self.b2) * (noisy_delta[k] ** 2)
            m_hat = self.m[k] / (1 - self.b1 ** self.t)
            v_hat = self.v[k] / (1 - self.b2 ** self.t)
            self.state[k] += self.lr * m_hat / (v_hat.sqrt() + self.eps)
        return self.state

def train_local_booster(x, y, num_class, num_boost_round=10, params=None):
    sample_weight = CLASS_WEIGHTS[y.astype(int)]
    dtrain = xgb.DMatrix(x, label=y, weight=sample_weight)
    p = {
        "objective": "multi:softprob",
        "num_class": num_class,
        "max_depth": 4,
        "eta": 0.3,
    }
    if params:
        p.update(params)
    booster = xgb.train(p, dtrain, num_boost_round=num_boost_round)
    return booster

def merge_boosters(boosters, num_class, num_boost_round):
    dumps = [json.loads(b.save_raw("json")) for b in boosters]

    base = copy.deepcopy(dumps[0])
    base_trees = base["learner"]["gradient_booster"]["model"]["trees"]
    base_tree_info = base["learner"]["gradient_booster"]["model"]["tree_info"]

    all_trees = list(base_trees)
    all_tree_info = list(base_tree_info)

    for d in dumps[1:]:
        trees = d["learner"]["gradient_booster"]["model"]["trees"]
        tree_info = d["learner"]["gradient_booster"]["model"]["tree_info"]
        for t in trees:
            t = copy.deepcopy(t)
            t["id"] = len(all_trees)
            all_trees.append(t)
        all_tree_info.extend(tree_info)

    base["learner"]["gradient_booster"]["model"]["trees"] = all_trees
    base["learner"]["gradient_booster"]["model"]["tree_info"] = all_tree_info

    n_total_trees = len(all_trees)
    base["learner"]["gradient_booster"]["model"]["gbtree_model_param"]["num_trees"] = str(n_total_trees)
    # num_parallel_tree stays 1 unless you're using boosted forests
    base["learner"]["learner_model_param"]["num_class"] = str(num_class)

    merged = xgb.Booster()
    merged.load_model(bytearray(json.dumps(base).encode("utf-8")))

    total_rounds = n_total_trees // num_class
    return merged, total_rounds #This sucked

def _leaf_value_arrays(trees):
    leaf_arrays = []
    for t in trees:
        lc = t["left_children"]
        sc = t["split_conditions"]
        arr = np.zeros(len(lc), dtype=np.float32)
        for i, left in enumerate(lc):
            if left == -1:
                arr[i] = sc[i]
        leaf_arrays.append(arr)
    return leaf_arrays

def get_tree_margins(booster, x, num_class, total_rounds):
    """
    Returns array of shape (N, total_rounds, num_class): the per-round
    (not cumulative) margin contribution for each class.
    Verified against predict(): sum of these per-round margins across all
    rounds + booster's base_score reconstructs predict(output_margin=True)
    to float32 precision.
    """
    dtest = xgb.DMatrix(x)
    leaf_idx = booster.predict(dtest, pred_leaf=True)  # (N, total_rounds * num_class)
    dump = json.loads(booster.save_raw("json"))
    trees = dump["learner"]["gradient_booster"]["model"]["trees"]
    leaf_arrays = _leaf_value_arrays(trees)

    N = x.shape[0]
    margins = np.zeros((N, total_rounds, num_class), dtype=np.float32)
    for t in range(total_rounds * num_class):
        r, c = divmod(t, num_class)
        margins[:, r, c] = leaf_arrays[t][leaf_idx[:, t].astype(np.int64)]
    return margins

def flatten_margins(margins):
    # (N, total_rounds, num_class) -> (N, total_rounds * num_class)
    N = margins.shape[0]
    return margins.reshape(N, -1)

def run_federated_xgboost(
    client_data, num_class, num_boost_round=10, rounds=20, lr=0.01, client_epochs=3,
):
    n_clients = len(client_data)
    client_sizes = [len(y) for _, y in client_data]

    # Phase 1: local trees -> merge (once)
    local_boosters = [train_local_booster(x, y, num_class, num_boost_round) for x, y in client_data]
    global_booster, total_rounds = merge_boosters(local_boosters, num_class, num_boost_round)

    # Precompute frozen-booster margin features per client
    client_feats = []
    for x, y in client_data:
        margins = get_tree_margins(global_booster, x, num_class, total_rounds)
        feats = flatten_margins(margins)
        client_feats.append((
            torch.tensor(feats, dtype=torch.float32),
            torch.tensor(y, dtype=torch.long),
        ))

    in_dim = total_rounds * num_class
    global_model = make_model(in_dim, num_class)
    server = FedAdamServer(global_model.state_dict(), lr=lr)  # <- your existing class

    # Phase 2: federated meta-learner training
    for r in range(rounds):
        client_states = []
        for x_feat, y_t in client_feats:
            local_model = make_model(in_dim, num_class)
            local_model.load_state_dict(server.state)
            state = client_update(local_model, x_feat, y_t, epochs=client_epochs, lr=lr)  # <- your existing fn
            client_states.append(state)
        server.step(client_states, client_sizes)

    final_model = make_model(in_dim, num_class)
    final_model.load_state_dict(server.state)
    return global_booster, final_model, total_rounds

class ClientDataset(torch.utils.data.Dataset):
    def __init__(self, folder):
        self.files = sorted(glob.glob(f'{folder}/*.pt'))
    def __len__(self):
        return len(self.files)
    def __getitem__(self, idx):
        x, y = torch.load(self.files[idx])
        return x, y, self.files[idx]  # bid recoverable from filename

In [ ]:
torch.manual_seed(0)
n_classes = 2  # binary: fraud / not fraud
cont_cols = num_cols  # global numeric columns fit in the client-feature step

class LinearWithEmbeddings(nn.Module):
    def __init__(self, num_cont, cat_cardinalities, emb_dim=8, n_classes=2):
        super().__init__()
        self.num_cont = num_cont
        self.embeddings = nn.ModuleList([nn.Embedding(card, emb_dim) for card in cat_cardinalities])
        in_dim = num_cont + emb_dim * len(cat_cardinalities)
        self.head = nn.Linear(in_dim, n_classes)

    def forward(self, x):
        x_cont = x[:, :self.num_cont].float()
        x_cat = x[:, self.num_cont:].long().clamp(min=0)
        embs = [emb(x_cat[:, i].clamp(0, emb.num_embeddings - 1))
                for i, emb in enumerate(self.embeddings)]
        x = torch.cat([x_cont] + embs, dim=-1)
        return self.head(x)


def evaluate(model, x, y):
    model.eval()
    with torch.no_grad():
        logits = model(x)
        loss = criterion(logits, y).item()
        acc = (logits.argmax(dim=1) == y).float().mean().item()
    model.train()   
    return loss, acc


In [ ]:
#old dead stuff from when we used timesteps
num_cont_per_step = len(num_cols) // N   # 16
num_cat_per_step  = len(cat_cols) // N   # 12

class CastedLinear(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.weight = nn.Parameter(torch.empty(out_dim, in_dim))
        nn.init.xavier_uniform_(self.weight)

    def forward(self, x):
        return F.linear(x, self.weight.to(x.dtype))


class MLP(nn.Module):
    def __init__(self, dim, mlp_mult):
        super().__init__()
        hidden = dim * mlp_mult
        self.gate = CastedLinear(dim, hidden)
        self.up   = CastedLinear(dim, hidden)
        self.proj = CastedLinear(hidden, dim)

    def forward(self, x):
        return self.proj(F.relu(self.gate(x)) * self.up(x))


class MLPClassifier(nn.Module):
    def __init__(self, num_cont, cat_cardinalities, dim=64, mlp_mult=4, depth=2, emb_dim=8):
        super().__init__()
        self.num_cont = num_cont
        self.num_cat = len(cat_cardinalities) 
        self.embeddings = nn.ModuleList([
            nn.Embedding(card, emb_dim) for card in cat_cardinalities
        ])
        in_dim = num_cont * N + emb_dim * len(cat_cardinalities)
        self.input_norm = nn.LayerNorm(in_dim)
        self.input_proj = CastedLinear(in_dim, dim)
        self.blocks = nn.ModuleList([MLP(dim, mlp_mult) for _ in range(depth)])
        self.head = nn.Linear(dim, 2)
        self.norms = nn.ModuleList([nn.LayerNorm(dim) for _ in range(depth)])

    def forward(self, x):
        x_cont = x[:, :self.num_cont * N].float()
        x_cat  = x[:, self.num_cont * N:].long().clamp(min=0)
        x_cat = x_cat.view(x.shape[0], N, self.num_cat)  # (B, N, 12)
        embs  = [e(x_cat[:, :, i].clamp(0, e.num_embeddings-1)).mean(dim=1)
                for i, e in enumerate(self.embeddings)]   # mean over timesteps → (B, emb_dim)
        x = torch.cat([x_cont] + embs, dim=-1)
        x = self.input_norm(x)
        x = self.input_proj(x)
        for block, norm in zip(self.blocks, self.norms):
            x = x + block(norm(x))
        return self.head(x).squeeze(-1)

def make_model():
    return MLPClassifier(len(cont_cols), cat_cardinalities, dim=128, mlp_mult=2, emb_dim=8)

In [ ]:

# build the held-out evaluation set with the same global encoders used per client 
test_group = test_df.copy()
test_group['__y_test__'] = y_test.loc[test_group.index].values
test_group = features_creator(test_group)
test_group['delta.t'] = np.log1p(test_group['delta.t'])

cat_x_eval = cat_pipe.transform(test_group[cat_cols]).astype(np.float32)
num_x_eval = num_scaler.transform(num_imputer.transform(test_group[num_cols])).astype(np.float32)

x_eval = torch.tensor(np.hstack([num_x_eval, cat_x_eval]), dtype=torch.float32)
y_eval = torch.tensor(test_group['__y_test__'].values, dtype=torch.long)

del test_group, cat_x_eval, num_x_eval
gc.collect()


35

In [ ]:
client_files = sorted(glob.glob('client_data/*.pt'))
print(f"{len(client_files)} clients found")

client_data = [
    (x, y, len(y))
    for x, y in (torch.load(f) for f in client_files)
]


1633 clients found


In [ ]:
# ---- Quick smoke test: federated XGBoost + meta-learner on a small subset ----

N_QUICK_CLIENTS = 20
QUICK_BOOST_ROUND = 2   # same value the full run uses
QUICK_EVAL_ROWS = 2000

quick_files = client_files[:N_QUICK_CLIENTS]
quick_client_data = [torch.load(f) for f in quick_files]  # [(x, y), ...]

quick_boosters = [
    train_local_booster(x.numpy(), y.numpy(), num_class=2, num_boost_round=QUICK_BOOST_ROUND)
    for x, y in quick_client_data
]
quick_global_booster, quick_total_rounds = merge_boosters(quick_boosters, num_class=2, num_boost_round=QUICK_BOOST_ROUND)
print(f"quick merged booster: {quick_total_rounds} rounds x 2 classes = {quick_total_rounds * 2} trees")

quick_in_dim = quick_total_rounds * 2

def make_quick_meta_model():
    return nn.Linear(quick_in_dim, 2)

quick_meta_data = []
for x, y in quick_client_data:
    margins = get_tree_margins(quick_global_booster, x.numpy(), num_class=2, total_rounds=quick_total_rounds)
    feats = torch.tensor(flatten_margins(margins), dtype=torch.float32)
    quick_meta_data.append((feats, y, len(y)))

x_eval_sub = x_eval[:QUICK_EVAL_ROWS]
y_eval_sub = y_eval[:QUICK_EVAL_ROWS]
quick_eval_margins = get_tree_margins(quick_global_booster, x_eval_sub.numpy(), num_class=2, total_rounds=quick_total_rounds)
quick_x_eval_meta = torch.tensor(flatten_margins(quick_eval_margins), dtype=torch.float32)

print("\n=== Quick test: FedAdam (meta-learner, subsampled) ===")
quick_model = make_quick_meta_model()
quick_server = FedAdamDPServer(quick_model.state_dict(), lr=1e-2, clip_norm=1.0, noise_sigma=0.0)

for rnd in range(5):
    client_states, sizes = [], []
    for x, y, n in quick_meta_data:
        m = make_quick_meta_model()
        m.load_state_dict(quick_server.state)
        client_states.append(client_update(m, x, y))
        sizes.append(n)
    quick_server.step(client_states, sizes)
    quick_model.load_state_dict(quick_server.state)
    loss, acc = evaluate(quick_model, quick_x_eval_meta, y_eval_sub)
    print(f"round {rnd}: test loss = {loss:.4f}, acc = {acc:.4f}")

del quick_client_data, quick_boosters, quick_meta_data, quick_eval_margins
gc.collect()

quick merged booster: 40 rounds x 2 classes = 80 trees

=== Quick test: FedAdam (meta-learner, subsampled) ===
round 0: test loss = 0.7434, acc = 0.7215
round 1: test loss = 0.7738, acc = 0.7450
round 2: test loss = 0.8045, acc = 0.7450
round 3: test loss = 0.8352, acc = 0.7450
round 4: test loss = 0.8652, acc = 0.7450


20

In [ ]:
# ---------- run FedAvg ----------
print("\n=== FedAvg ===")
global_model = make_model()
global_state = global_model.state_dict()

for rnd in range(50):
    client_states, sizes = [], []
    for x, y, n in client_data:
        m = make_model()
        m.load_state_dict(global_state)
        client_states.append(client_update(m, x, y))
        sizes.append(n)
    global_state = fedavg_aggregate(client_states, sizes)
    sample_x, sample_y, sample_n = client_data[0]
    global_model.load_state_dict(global_state)
    loss, acc = evaluate(global_model,x_eval,y_eval)
    print(f"round {rnd}: test loss = {loss:.4f}, acc = {acc:.4f}")

In [ ]:
def torch_predict_proba(model, x):
    model.eval()
    with torch.no_grad():
        proba = F.softmax(model(x), dim=1).numpy()
    model.train()
    return proba

def report_torch_model(name, model, x, y_true_tensor):
    proba = torch_predict_proba(model, x)
    preds = proba.argmax(axis=1)
    y_true = y_true_tensor.numpy()
    print(f"\n=== {name} ===")
    print(classification_report(y_true, preds))
    print(f"ROC-AUC: {roc_auc_score(y_true, proba[:, 1]):.4f}")
    return preds, proba


In [ ]:
os.makedirs("models_H2", exist_ok=True)
report_torch_model("FedAvg", global_model, x_eval, y_eval)
torch.save(global_model.state_dict(), "models_H2/mlp_fedavg.pt")

In [ ]:
# ---------- run FedAdam ----------
print("\n=== FedAdam ===")
global_model2 = make_model()
server = FedAdamServer(global_model2.state_dict(), lr=1e-2, clip_norm=1.0)

for rnd in range(50):
    client_states, sizes = [], []
    for x, y, n in client_data:
        m = make_model()
        m.load_state_dict(server.state)
        client_states.append(client_update(m, x, y))
        sizes.append(n)
    server.step(client_states, sizes)
    global_model2.load_state_dict(server.state)
    loss, acc = evaluate(global_model2, x_eval, y_eval)
    print(f"round {rnd}: test loss = {loss:.4f}, acc = {acc:.4f}")


In [ ]:
report_torch_model("FedAdam", global_model2, x_eval, y_eval)
torch.save(global_model2.state_dict(), "models_H2/mlp_fedadam.pt")

In [ ]:

# ---------- run FedAdam + Noise  ----------
print("\n=== FedAdam + DP noise ===")
global_model3 = make_model()
dp_server = FedAdamDPServer(global_model3.state_dict(), lr=1e-2, clip_norm=1.0, noise_sigma=0.1)

for rnd in range(50):
    client_states, sizes = [], []
    for x, y, n in client_data:
        m = make_model()
        m.load_state_dict(dp_server.state)
        client_states.append(client_update(m, x, y))
        sizes.append(n)
    dp_server.step(client_states, sizes)
    global_model3.load_state_dict(dp_server.state)
    loss, acc = evaluate(global_model3, x_eval, y_eval)
    print(f"round {rnd}: test loss = {loss:.4f}, acc = {acc:.4f}")

In [ ]:
report_torch_model("FedAdam+DP noise", global_model3, x_eval, y_eval)
torch.save(global_model3.state_dict(), "models_H2/mlp_fedadam_dp.pt")

In [16]:

# ---------- Phase 1: local XGBoost training + tree merge (once) ----------
print("\n=== Federated XGBoost: Phase 1 (tree bagging) ===")
num_class = 2
num_boost_round = 3

local_boosters = [
    train_local_booster(x.numpy(), y.numpy(), num_class, num_boost_round)
    for x, y, n in client_data
]
global_booster, total_rounds = merge_boosters(local_boosters, num_class, num_boost_round)
print(f"merged booster: {total_rounds} rounds x {num_class} classes = "
      f"{total_rounds * num_class} trees")

# ---------- Precompute frozen-booster margin features per client ----------
in_dim = total_rounds * num_class


=== Federated XGBoost: Phase 1 (tree bagging) ===
merged booster: 4899 rounds x 2 classes = 9798 trees


In [17]:
joblib.dump({
    "global_booster": global_booster,       # xgb.Booster is picklable
    "total_rounds": total_rounds,
    "num_class": num_class,
    "in_dim": in_dim,
}, "federated_xgboost_margins.joblib")

['federated_xgboost_margins.joblib']

In [ ]:
cache = joblib.load("federated_xgboost_margins.joblib")
global_booster = cache["global_booster"]
total_rounds = cache["total_rounds"]
num_class = cache["num_class"]
in_dim = cache["in_dim"]

def make_meta_model():
    return nn.Linear(in_dim, 2)

os.makedirs('client_data_meta', exist_ok=True) 
# We get alll the tree preds
for f in glob.glob('client_data/*.pt'):
    bid = os.path.splitext(os.path.basename(f))[0]
    out_path = f'client_data_meta/{bid}.pt'
    if os.path.exists(out_path):
        continue
    x, y = torch.load(f)
    margins = get_tree_margins(global_booster, x.numpy(), num_class, total_rounds)
    feats = torch.tensor(flatten_margins(margins), dtype=torch.float32)
    tmp_path = out_path + '.tmp'
    torch.save((feats, y), tmp_path)
    os.replace(tmp_path, out_path)
    del x, y, margins, feats
    gc.collect()

x_eval_margins = get_tree_margins(global_booster, x_eval.numpy(), num_class, total_rounds)
x_eval_meta = torch.tensor(flatten_margins(x_eval_margins), dtype=torch.float32)
del x_eval_margins
gc.collect()

meta_files = sorted(glob.glob('client_data_meta/*.pt'))
meta_sizes = [torch.load(f)[1].shape[0] for f in meta_files]  # just the label count, cheap

# ---------- Phase 2: run FedAdam ----------
print("\n=== Federated XGBoost: FedAdam + Linear (meta-learner) ===")
meta_model = make_meta_model()  
dp_server = FedAdamServer(meta_model.state_dict(), lr=1e-2, clip_norm=1.0)

for rnd in range(50):
    client_states, sizes = [], []
    for f, n in zip(meta_files, meta_sizes):
        x, y = torch.load(f)
        m = make_meta_model()
        m.load_state_dict(dp_server.state)
        client_states.append(client_update(m, x, y))
        sizes.append(n)
        del x, y
    dp_server.step(client_states, sizes)
    meta_model.load_state_dict(dp_server.state)
    loss, acc = evaluate(meta_model, x_eval_meta, y_eval)
    print(f"round {rnd}: test loss = {loss:.4f}, acc = {acc:.4f}")

os.makedirs("models_H2", exist_ok=True)
torch.save(meta_model.state_dict(), "models_H2/xgb_meta_learner.pt")
joblib.dump(
    {"global_booster": global_booster, "total_rounds": total_rounds,
     "num_class": num_class, "in_dim": in_dim},
    "models_H2/xgb_global_booster.joblib",
)


In [ ]:
# ---- Load and test the saved XGBoost meta-learner ----
loaded = joblib.load("models_H2/xgb_global_booster.joblib")
loaded_booster = loaded["global_booster"]
loaded_total_rounds = loaded["total_rounds"]
loaded_num_class = loaded["num_class"]
loaded_in_dim = loaded["in_dim"]

loaded_meta_model = nn.Linear(loaded_in_dim, loaded_num_class)
loaded_meta_model.load_state_dict(torch.load("models_H2/xgb_meta_learner.pt"))
loaded_meta_model.eval()

x_eval_margins_check = get_tree_margins(loaded_booster, x_eval.numpy(), loaded_num_class, loaded_total_rounds)
x_eval_meta_check = torch.tensor(flatten_margins(x_eval_margins_check), dtype=torch.float32)
del x_eval_margins_check

report_torch_model("XGBoost meta-learner (loaded from disk)", loaded_meta_model, x_eval_meta_check, y_eval)